# ee.ipynb — 전체 통합 CCTV 이상행동 분석 파이프라인

이 노트북은 기존 통합 방향을 유지하면서, **싸움/스토킹 판단부만 더 강하게 교체한 후속 버전**입니다.

처리 대상 label은 다음 전체 범위를 포함합니다.

- `fight` : 싸움
- `stalking` : 스토킹/추적
- `sidewalk_riding` : 이륜차 인도 주행
- `no_helmet` : 헬멧 미착용
- `electric_kickboard` : 전동 킥보드/스쿠터 계열
- `intrusion` : 침입/경계선 침범
- `other` : 위 조건을 통과하지 못한 기타

핵심 변경점은 **사람 2명 이상이 등장했을 때 fight와 stalking 모두 같은 track-pair box 구조로 분석**하고, 영상 전체 평균이 아니라 **4초 sliding window**의 최대 의심 구간을 사용한다는 점입니다. 따라서 영상 초반이 평범하고 몇 초 뒤 이상행동이 시작되는 경우도 점수에 반영됩니다.

최종 표의 `final_label`은 영상당 **반드시 1개만** 출력됩니다. 내부 후보 점수들은 JSON에 보존하지만, 최종 판정은 가중치와 gate 조건을 적용한 뒤 가장 강한 하나만 선택합니다.


## 0. 설치 안내

Jupyter 안에서 설치하려면 아래 셀의 주석을 풀고 실행하세요.

CMD에서 설치하려면 다음 명령을 사용하면 됩니다.

```bash
pip install ultralytics opencv-python pandas tqdm numpy transformers torch pillow
```

`transformers/torch`는 SegFormer 기반 인도/도로 segmentation을 사용할 때 필요합니다. 설치되어 있지 않으면 자동으로 segmentation 없이 진행하고, 인도 주행은 `unknown`에 가깝게 보수적으로 처리합니다.


In [ ]:
# 필요하면 주석을 풀고 한 번만 실행하세요.
# !pip install ultralytics opencv-python pandas tqdm numpy transformers torch pillow


In [1]:
import os
import json
import math
import time
import traceback
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations

import cv2
import numpy as np
from tqdm.auto import tqdm

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except Exception:
    display = None

try:
    from ultralytics import YOLO
except ImportError as e:
    raise ImportError(
        "ultralytics가 설치되어 있지 않습니다. CMD 또는 설치 셀에서 `pip install ultralytics`를 먼저 실행하세요."
    ) from e

try:
    from ultralytics import YOLOWorld
except Exception:
    YOLOWorld = None

print("imports ok")


imports ok


## 1. 경로 설정

기본 후보 경로에는 지금까지 사용하던 Windows 경로를 넣어두었습니다.

- 영상 폴더 기본 후보: `C:\Users\우성한\Sample (1)\Sample`
- 출력 폴더 기본값: `C:\Users\우성한\outputs\prediction\ee`

경로가 다르면 `VIDEO_ROOT`와 `OUTPUT_ROOT`만 직접 바꾸면 됩니다.


In [2]:
# =========================
# 경로 설정
# =========================
VIDEO_ROOT_CANDIDATES = [
    Path(r"C:\Users\우성한\Sample (1)\Sample"),
    Path(r"C:\Users\우성한\Sample"),
    Path(r".\Sample"),
    Path(r".\sample"),
    Path("/mnt/data"),  # 업로드 테스트용
]

# 필요하면 여기 직접 지정하세요.
VIDEO_ROOT = None
for p in VIDEO_ROOT_CANDIDATES:
    if p.exists():
        VIDEO_ROOT = p
        break
if VIDEO_ROOT is None:
    VIDEO_ROOT = VIDEO_ROOT_CANDIDATES[0]

OUTPUT_ROOT = Path(r"C:\Users\우성한\outputs\prediction\ee")
if str(VIDEO_ROOT).startswith("/mnt/data"):
    OUTPUT_ROOT = Path("/mnt/data/outputs/prediction/ee")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DETAIL_DIR = OUTPUT_ROOT / "details"
DETAIL_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".wmv"}

print("VIDEO_ROOT:", VIDEO_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("exists:", VIDEO_ROOT.exists())


VIDEO_ROOT: C:\Users\우성한\Sample (1)\Sample
OUTPUT_ROOT: C:\Users\우성한\outputs\prediction\ee
exists: True


## 2. 분석 설정

이번 버전은 전체 이상행동 label을 유지하면서, fight/stalking만 강화합니다.

- fight/stalking: 사람 track pair + pair box + sliding window
- sidewalk riding: 이륜차/탑승자 + SegFormer 또는 사용자 polygon 기반 인도 판정
- no helmet: 전용 helmet 모델이 있으면 확정, 없으면 YOLO-World 보조 결과만 제한적으로 사용
- electric kickboard: YOLO-World 또는 전용 모델 class name 기반 탐지
- intrusion: 사용자 지정 경계선/제한구역 polygon + 사람 foot point crossing 기반


In [3]:
# =========================
# 모델 설정
# =========================
DETECT_MODEL_NAME = "yolo11n.pt"        # 사람/자전거/오토바이 기본 탐지
POSE_MODEL_NAME = "yolo11n-pose.pt"     # 싸움 강화용 pose. 실패하면 bbox motion fallback

USE_POSE = True
POSE_EVERY_N_SAMPLED_FRAMES = 2

# YOLO-World는 전동킥보드/헬멧/문/펜스 등 COCO에 없는 객체를 보강합니다.
# 설치/다운로드가 안 되면 자동으로 비활성화됩니다.
USE_YOLO_WORLD = True
WORLD_MODEL_NAME = "yolov8s-worldv2.pt"
WORLD_CLASSES = [
    "person",
    "bicycle",
    "motorcycle",
    "motorbike",
    "electric scooter",
    "kickboard",
    "scooter",
    "helmet",
    "no helmet",
    "person without helmet",
    "fence",
    "gate",
    "door",
    "railing",
    "barrier",
]

# 헬멧 전용 모델이 있으면 여기에 경로를 넣으세요.
# class 이름 예시: helmet, no_helmet, no helmet, without helmet
HELMET_MODEL_PATH = None  # 예: r"C:\models\helmet_best.pt"

# 전동킥보드 전용 모델이 있으면 여기에 경로를 넣으세요.
KICKBOARD_MODEL_PATH = None  # 예: r"C:\models\kickboard_best.pt"

# =========================
# SegFormer 인도/도로 segmentation 설정
# =========================
USE_SEGFORMER = True
SEG_MODEL_ID_CPU = "nvidia/segformer-b0-finetuned-ade-512-512"
SEG_MODEL_ID_GPU = "nvidia/segformer-b0-finetuned-ade-512-512"
SEG_EVERY_N_SAMPLED_FRAMES = 3
SEG_POINT_RADIUS = 8

# segmentation이 너무 느리면 False로 바꾸세요.
# USE_SEGFORMER = False

# =========================
# 사용자 지정 ROI / 침입 경계 설정
# =========================
# 인도 영역을 직접 알고 있으면 polygon을 지정하면 SegFormer보다 안정적입니다.
# 형식: [[(x1,y1), (x2,y2), ...], ...]
SIDEWALK_POLYGONS = []
ROAD_POLYGONS = []

# 침입 경계선: 사람이 이 선을 건너면 intrusion 후보가 됩니다.
# 형식: [{"name":"gate_line", "p1":(x1,y1), "p2":(x2,y2)}]
INTRUSION_LINES = []

# 제한구역 polygon: foot point가 이 영역 안으로 들어가면 intrusion 후보가 됩니다.
# 형식: [{"name":"restricted_area", "polygon":[(x1,y1), (x2,y2), ...]}]
RESTRICTED_POLYGONS = []

# =========================
# 영상 샘플링 / 윈도우 설정
# =========================
ANALYZE_FPS = 3.0
MAX_SECONDS_PER_VIDEO = None  # 테스트할 때만 예: 40. 전체 처리면 None
WINDOW_SEC = 4.0
WINDOW_STEP_SEC = 2.0
MIN_WINDOW_OBS = 4

# =========================
# YOLO confidence 설정
# =========================
YOLO_IMGSZ = 640
YOLO_CONF = 0.22
PERSON_CONF = 0.25
TWOWHEELER_CONF = 0.20
WORLD_CONF = 0.18
HELMET_CONF = 0.20

# =========================
# track 설정
# =========================
TRACK_MAX_GAP_SEC = 1.2
TRACK_IOU_WEIGHT = 0.65
TRACK_DIST_WEIGHT = 0.35
TRACK_MATCH_SCORE_TH = 0.25
MIN_PAIR_OBS = 4

# =========================
# 최종 label 가중치와 threshold
# =========================
LABEL_WEIGHTS = {
    "fight": 1.35,
    "stalking": 1.18,
    "sidewalk_riding": 1.12,
    "no_helmet": 1.18,
    "electric_kickboard": 1.12,
    "intrusion": 1.25,
    "other": 0.0,
}

LABEL_THRESHOLDS = {
    "fight": 0.42,
    "stalking": 0.46,
    "sidewalk_riding": 0.48,
    "no_helmet": 0.52,
    "electric_kickboard": 0.42,
    "intrusion": 0.52,
}

# COCO class id
CLS_PERSON = 0
CLS_BICYCLE = 1
CLS_MOTORCYCLE = 3
COCO_TWOWHEELER_CLASSES = {CLS_BICYCLE, CLS_MOTORCYCLE}

LABEL_KO = {
    "fight": "싸움",
    "stalking": "스토킹",
    "sidewalk_riding": "이륜차 인도 주행",
    "no_helmet": "헬멧 미착용",
    "electric_kickboard": "전동 킥보드",
    "intrusion": "침입",
    "other": "기타",
}

print("settings ready")


settings ready


## 3. 공통 유틸 함수


In [4]:
def safe_div(a, b, default=0.0):
    return float(a) / float(b) if b else default


def clamp(v, lo=0.0, hi=1.0):
    return max(lo, min(hi, float(v)))


def box_xyxy(box):
    x1, y1, x2, y2 = map(float, box)
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    return [x1, y1, x2, y2]


def box_area(box):
    x1, y1, x2, y2 = box_xyxy(box)
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def box_center(box):
    x1, y1, x2, y2 = box_xyxy(box)
    return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0], dtype=np.float32)


def box_foot(box):
    x1, y1, x2, y2 = box_xyxy(box)
    return np.array([(x1 + x2) / 2.0, y2], dtype=np.float32)


def box_wh(box):
    x1, y1, x2, y2 = box_xyxy(box)
    return max(1.0, x2 - x1), max(1.0, y2 - y1)


def box_iou(a, b):
    ax1, ay1, ax2, ay2 = box_xyxy(a)
    bx1, by1, bx2, by2 = box_xyxy(b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    union = box_area(a) + box_area(b) - inter
    return safe_div(inter, union, 0.0)


def expand_box(box, ratio=0.25, image_shape=None):
    x1, y1, x2, y2 = box_xyxy(box)
    w, h = x2 - x1, y2 - y1
    nx1, ny1 = x1 - w * ratio, y1 - h * ratio
    nx2, ny2 = x2 + w * ratio, y2 + h * ratio
    if image_shape is not None:
        H, W = image_shape[:2]
        nx1, nx2 = max(0, nx1), min(W - 1, nx2)
        ny1, ny2 = max(0, ny1), min(H - 1, ny2)
    return [float(nx1), float(ny1), float(nx2), float(ny2)]


def union_box(boxes):
    boxes = [box_xyxy(b) for b in boxes if b is not None]
    if not boxes:
        return None
    arr = np.array(boxes, dtype=np.float32)
    return [float(arr[:,0].min()), float(arr[:,1].min()), float(arr[:,2].max()), float(arr[:,3].max())]


def point_in_box(point, box):
    x, y = float(point[0]), float(point[1])
    x1, y1, x2, y2 = box_xyxy(box)
    return x1 <= x <= x2 and y1 <= y <= y2


def point_in_poly(point, polygon):
    if not polygon:
        return False
    poly = np.array(polygon, dtype=np.int32)
    return cv2.pointPolygonTest(poly, (float(point[0]), float(point[1])), False) >= 0


def orientation(a, b, c):
    return (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])


def segment_crossed(p_prev, p_cur, a, b):
    # p_prev -> p_cur 선분이 a-b 선분을 건너는지 확인
    p_prev = np.array(p_prev, dtype=np.float32)
    p_cur = np.array(p_cur, dtype=np.float32)
    a = np.array(a, dtype=np.float32)
    b = np.array(b, dtype=np.float32)
    o1 = orientation(p_prev, p_cur, a)
    o2 = orientation(p_prev, p_cur, b)
    o3 = orientation(a, b, p_prev)
    o4 = orientation(a, b, p_cur)
    return (o1 * o2 < 0) and (o3 * o4 < 0)


def normalize_name(name):
    return str(name).lower().replace("_", " ").replace("-", " ").strip()


def is_twowheeler_name(name):
    n = normalize_name(name)
    keys = ["bicycle", "bike", "motorcycle", "motorbike", "scooter", "kickboard"]
    return any(k in n for k in keys)


def is_kickboard_name(name):
    n = normalize_name(name)
    keys = ["electric scooter", "kickboard", "scooter", "e scooter"]
    return any(k in n for k in keys)


def is_helmet_name(name):
    n = normalize_name(name)
    return "helmet" in n and "no" not in n and "without" not in n


def is_no_helmet_name(name):
    n = normalize_name(name)
    return ("no helmet" in n) or ("without helmet" in n) or ("nohelmet" in n)


def is_barrier_name(name):
    n = normalize_name(name)
    keys = ["fence", "gate", "door", "railing", "barrier", "wall"]
    return any(k in n for k in keys)


def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {"ok": False}
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    duration = safe_div(frame_count, fps, 0.0)
    cap.release()
    return {
        "ok": True,
        "fps": float(fps),
        "frame_count": int(frame_count),
        "width": width,
        "height": height,
        "duration": float(duration),
    }


def iter_sampled_frames(video_path, analyze_fps=3.0, max_seconds=None):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    step = max(1, int(round(fps / analyze_fps)))
    max_frame = total_frames
    if max_seconds is not None:
        max_frame = min(max_frame, int(max_seconds * fps))

    frame_idx = 0
    sampled_idx = 0
    while frame_idx < max_frame:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break
        t = frame_idx / fps
        yield {
            "frame": frame,
            "frame_idx": int(frame_idx),
            "sampled_idx": int(sampled_idx),
            "t": float(t),
            "fps": float(fps),
        }
        sampled_idx += 1
        frame_idx += step
    cap.release()


## 4. 모델 로드

없는 모델은 자동으로 비활성화하고, 가능한 모델만 사용합니다.


In [5]:
print("loading detection model:", DETECT_MODEL_NAME)
det_model = YOLO(DETECT_MODEL_NAME)

pose_model = None
if USE_POSE:
    try:
        print("loading pose model:", POSE_MODEL_NAME)
        pose_model = YOLO(POSE_MODEL_NAME)
    except Exception as e:
        print("pose model 로드 실패. bbox motion fallback으로 진행합니다:", e)
        pose_model = None

world_model = None
if USE_YOLO_WORLD and YOLOWorld is not None:
    try:
        print("loading YOLO-World model:", WORLD_MODEL_NAME)
        world_model = YOLOWorld(WORLD_MODEL_NAME)
        world_model.set_classes(WORLD_CLASSES)
        print("YOLO-World classes:", WORLD_CLASSES)
    except Exception as e:
        print("YOLO-World 로드 실패. COCO 기본 탐지만 사용합니다:", e)
        world_model = None
else:
    print("YOLO-World 비활성화 또는 ultralytics YOLOWorld 미지원")

helmet_model = None
if HELMET_MODEL_PATH is not None and Path(HELMET_MODEL_PATH).exists():
    try:
        print("loading helmet model:", HELMET_MODEL_PATH)
        helmet_model = YOLO(str(HELMET_MODEL_PATH))
    except Exception as e:
        print("helmet model 로드 실패:", e)
        helmet_model = None
else:
    print("helmet 전용 모델 없음: no_helmet은 보수적으로 판단합니다.")

kickboard_model = None
if KICKBOARD_MODEL_PATH is not None and Path(KICKBOARD_MODEL_PATH).exists():
    try:
        print("loading kickboard model:", KICKBOARD_MODEL_PATH)
        kickboard_model = YOLO(str(KICKBOARD_MODEL_PATH))
    except Exception as e:
        print("kickboard model 로드 실패:", e)
        kickboard_model = None
else:
    print("kickboard 전용 모델 없음: YOLO-World/COCO 이름 기반으로 보조 판단합니다.")


loading detection model: yolo11n.pt
loading pose model: yolo11n-pose.pt
loading YOLO-World model: yolov8s-worldv2.pt
YOLO-World classes: ['person', 'bicycle', 'motorcycle', 'motorbike', 'electric scooter', 'kickboard', 'scooter', 'helmet', 'no helmet', 'person without helmet', 'fence', 'gate', 'door', 'railing', 'barrier']
helmet 전용 모델 없음: no_helmet은 보수적으로 판단합니다.
kickboard 전용 모델 없음: YOLO-World/COCO 이름 기반으로 보조 판단합니다.


In [6]:
# SegFormer 로드. 실패하면 None으로 두고 polygon 기반 또는 unknown 처리.
seg_processor = None
seg_model = None
seg_id2label = {}
seg_device = None

if USE_SEGFORMER:
    try:
        import torch
        from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
        seg_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        seg_model_id = SEG_MODEL_ID_GPU if seg_device.type == "cuda" else SEG_MODEL_ID_CPU
        print("loading SegFormer:", seg_model_id, "device:", seg_device)
        seg_processor = SegformerImageProcessor.from_pretrained(seg_model_id)
        seg_model = SegformerForSemanticSegmentation.from_pretrained(seg_model_id).to(seg_device)
        seg_model.eval()
        seg_id2label = {int(k): v for k, v in seg_model.config.id2label.items()}
        print("SegFormer loaded")
    except Exception as e:
        print("SegFormer 로드 실패. 인도 주행은 polygon 또는 unknown으로 처리합니다:", e)
        seg_processor = None
        seg_model = None
        seg_id2label = {}
else:
    print("SegFormer disabled")


loading SegFormer: nvidia/segformer-b0-finetuned-ade-512-512 device: cuda


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

C:\Users\우성한\cv_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\우성한\.cache\huggingface\hub\models--nvidia--segformer-b0-finetuned-ade-512-512. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

SegFormer loaded


## 5. 탐지 / 포즈 / segmentation 함수


In [7]:
def yolo_result_to_objects(result, names=None, source="det"):
    objs = []
    if result is None or not hasattr(result, "boxes") or result.boxes is None:
        return objs
    if names is None:
        names = getattr(result, "names", {}) or {}
    for b in result.boxes:
        try:
            cls_id = int(b.cls[0].item())
            conf = float(b.conf[0].item())
            xyxy = b.xyxy[0].detach().cpu().numpy().astype(float).tolist()
        except Exception:
            continue
        class_name = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
        objs.append({
            "box": box_xyxy(xyxy),
            "conf": conf,
            "class_id": cls_id,
            "class_name": str(class_name),
            "source": source,
        })
    return objs


def run_base_detector(frame):
    res = det_model.predict(frame, imgsz=YOLO_IMGSZ, conf=YOLO_CONF, verbose=False)[0]
    return yolo_result_to_objects(res, names=det_model.names, source="coco")


def run_world_detector(frame):
    if world_model is None:
        return []
    try:
        res = world_model.predict(frame, imgsz=YOLO_IMGSZ, conf=WORLD_CONF, verbose=False)[0]
        return yolo_result_to_objects(res, names=world_model.names, source="world")
    except Exception:
        return []


def run_extra_model(frame, model, source):
    if model is None:
        return []
    try:
        res = model.predict(frame, imgsz=YOLO_IMGSZ, conf=YOLO_CONF, verbose=False)[0]
        return yolo_result_to_objects(res, names=model.names, source=source)
    except Exception:
        return []


def merge_objects(objects, iou_th=0.80):
    # 같은 class/source에 가까운 중복 bbox 제거. world와 coco는 둘 다 보존하되 매우 중복이면 conf 높은 것만 남김.
    objects = sorted(objects, key=lambda o: o.get("conf", 0), reverse=True)
    kept = []
    for o in objects:
        dup = False
        for k in kept:
            same_name = normalize_name(o.get("class_name")) == normalize_name(k.get("class_name"))
            same_type = (
                (is_twowheeler_name(o.get("class_name")) and is_twowheeler_name(k.get("class_name"))) or
                (is_barrier_name(o.get("class_name")) and is_barrier_name(k.get("class_name"))) or
                (o.get("class_id") == k.get("class_id") and o.get("source") == k.get("source"))
            )
            if same_type and box_iou(o["box"], k["box"]) >= iou_th:
                dup = True
                break
        if not dup:
            kept.append(o)
    return kept


def detect_all_objects(frame):
    objs = []
    objs.extend(run_base_detector(frame))
    objs.extend(run_world_detector(frame))
    objs.extend(run_extra_model(frame, kickboard_model, "kickboard_model"))
    # helmet_model은 crop용으로 주로 쓰지만 전체 프레임에서 잡히는 경우도 보조로 사용
    objs.extend(run_extra_model(frame, helmet_model, "helmet_model_full"))
    return merge_objects(objs)


def detect_persons_from_objects(objects):
    persons = []
    for o in objects:
        cname = normalize_name(o.get("class_name"))
        is_person = (o.get("class_id") == CLS_PERSON and o.get("source") == "coco") or cname == "person"
        if is_person and o.get("conf", 0) >= PERSON_CONF:
            persons.append({
                "box": o["box"],
                "conf": float(o.get("conf", 0)),
                "class_name": "person",
                "pose": None,
            })
    return persons


def yolo_pose_persons(frame):
    if pose_model is None:
        return []
    try:
        res = pose_model.predict(frame, imgsz=YOLO_IMGSZ, conf=PERSON_CONF, verbose=False)[0]
    except Exception:
        return []

    out = []
    boxes = res.boxes
    kpts = getattr(res, "keypoints", None)
    if boxes is None:
        return out
    for i, b in enumerate(boxes):
        try:
            conf = float(b.conf[0].item())
            xyxy = b.xyxy[0].detach().cpu().numpy().astype(float).tolist()
            pose_xy = None
            pose_conf = None
            if kpts is not None and kpts.xy is not None:
                pose_xy = kpts.xy[i].detach().cpu().numpy().astype(float)
                if getattr(kpts, "conf", None) is not None:
                    pose_conf = kpts.conf[i].detach().cpu().numpy().astype(float)
            out.append({"box": box_xyxy(xyxy), "conf": conf, "pose_xy": pose_xy, "pose_conf": pose_conf})
        except Exception:
            continue
    return out


def attach_pose_to_persons(person_dets, pose_dets):
    if not pose_dets:
        for p in person_dets:
            p["pose"] = None
        return person_dets
    used = set()
    for p in person_dets:
        best_i, best_score = None, 0.0
        for i, q in enumerate(pose_dets):
            if i in used:
                continue
            s = box_iou(p["box"], q["box"])
            if s > best_score:
                best_score = s
                best_i = i
        if best_i is not None and best_score > 0.20:
            used.add(best_i)
            q = pose_dets[best_i]
            p["pose"] = {"xy": q.get("pose_xy"), "conf": q.get("pose_conf"), "box_iou": float(best_score)}
        else:
            p["pose"] = None
    return person_dets


def surface_from_polygons(point):
    for poly in SIDEWALK_POLYGONS:
        if point_in_poly(point, poly):
            return "sidewalk"
    for poly in ROAD_POLYGONS:
        if point_in_poly(point, poly):
            return "road"
    return "unknown"


def segment_frame_surface(frame):
    if seg_model is None or seg_processor is None:
        return None
    try:
        import torch
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        inputs = seg_processor(images=rgb, return_tensors="pt")
        inputs = {k: v.to(seg_device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = seg_model(**inputs)
        logits = outputs.logits
        up = torch.nn.functional.interpolate(
            logits,
            size=rgb.shape[:2],
            mode="bilinear",
            align_corners=False,
        )
        pred = up.argmax(dim=1)[0].detach().cpu().numpy().astype(np.int32)
        return pred
    except Exception:
        return None


def label_to_surface(label_name):
    n = normalize_name(label_name)
    sidewalk_keys = ["sidewalk", "pavement", "path", "walkway", "floor", "earth"]
    road_keys = ["road", "street", "runway"]
    if any(k in n for k in sidewalk_keys):
        return "sidewalk"
    if any(k in n for k in road_keys):
        return "road"
    return "unknown"


def surface_at_point(seg_mask, point):
    # polygon이 있으면 polygon이 우선입니다.
    poly_s = surface_from_polygons(point)
    if poly_s != "unknown":
        return poly_s
    if seg_mask is None:
        return "unknown"
    H, W = seg_mask.shape[:2]
    x, y = int(round(point[0])), int(round(point[1]))
    if not (0 <= x < W and 0 <= y < H):
        return "unknown"
    r = max(1, int(SEG_POINT_RADIUS))
    x1, x2 = max(0, x - r), min(W, x + r + 1)
    y1, y2 = max(0, y - r), min(H, y + r + 1)
    patch = seg_mask[y1:y2, x1:x2]
    if patch.size == 0:
        return "unknown"
    ids, counts = np.unique(patch, return_counts=True)
    best = int(ids[np.argmax(counts)])
    label = seg_id2label.get(best, str(best))
    return label_to_surface(label)


## 6. 사람 track 생성

싸움과 스토킹 모두 같은 사람 track을 사용합니다. 사람이 2명 이상이면 매 프레임 pair box를 만들고, sliding window에서 pair 단위 feature를 계산합니다.


In [8]:
def match_track_score(det, track, t):
    gap = t - track.get("last_t", -1e9)
    if gap < -1e-6 or gap > TRACK_MAX_GAP_SEC:
        return -1.0
    iou = box_iou(det["box"], track["box"])
    det_c = box_center(det["box"])
    tr_c = box_center(track["box"])
    _, h = box_wh(track["box"])
    dist = float(np.linalg.norm(det_c - tr_c))
    dist_score = math.exp(-dist / max(30.0, h * 1.5))
    return TRACK_IOU_WEIGHT * iou + TRACK_DIST_WEIGHT * dist_score


def update_person_tracks(person_dets, tracks, next_track_id, t):
    assigned_tracks = set()
    assigned_dets = set()
    pairs = []
    for di, det in enumerate(person_dets):
        for tid, tr in tracks.items():
            if tid in assigned_tracks:
                continue
            s = match_track_score(det, tr, t)
            if s >= TRACK_MATCH_SCORE_TH:
                pairs.append((s, di, tid))
    pairs.sort(reverse=True, key=lambda x: x[0])

    frame_persons = []
    for s, di, tid in pairs:
        if di in assigned_dets or tid in assigned_tracks:
            continue
        det = person_dets[di]
        tr = tracks[tid]
        prev_center = np.array(tr.get("center", box_center(det["box"])), dtype=np.float32)
        prev_foot = np.array(tr.get("foot", box_foot(det["box"])), dtype=np.float32)
        dt = max(1e-3, t - tr.get("last_t", t))
        center = box_center(det["box"])
        foot = box_foot(det["box"])
        velocity = (center - prev_center) / dt
        foot_velocity = (foot - prev_foot) / dt
        person = {
            "track_id": int(tid),
            "box": det["box"],
            "conf": float(det.get("conf", 0)),
            "center": center.tolist(),
            "foot": foot.tolist(),
            "velocity": velocity.tolist(),
            "foot_velocity": foot_velocity.tolist(),
            "speed_px_s": float(np.linalg.norm(velocity)),
            "pose": det.get("pose"),
            "track_match_score": float(s),
        }
        tracks[tid].update({
            "box": det["box"],
            "center": center.tolist(),
            "foot": foot.tolist(),
            "last_t": float(t),
            "velocity": velocity.tolist(),
            "foot_velocity": foot_velocity.tolist(),
            "age": tr.get("age", 0) + 1,
        })
        assigned_dets.add(di)
        assigned_tracks.add(tid)
        frame_persons.append(person)

    for di, det in enumerate(person_dets):
        if di in assigned_dets:
            continue
        tid = next_track_id
        next_track_id += 1
        center = box_center(det["box"])
        foot = box_foot(det["box"])
        tracks[tid] = {
            "box": det["box"],
            "center": center.tolist(),
            "foot": foot.tolist(),
            "last_t": float(t),
            "velocity": [0.0, 0.0],
            "foot_velocity": [0.0, 0.0],
            "age": 1,
        }
        person = {
            "track_id": int(tid),
            "box": det["box"],
            "conf": float(det.get("conf", 0)),
            "center": center.tolist(),
            "foot": foot.tolist(),
            "velocity": [0.0, 0.0],
            "foot_velocity": [0.0, 0.0],
            "speed_px_s": 0.0,
            "pose": det.get("pose"),
            "track_match_score": 1.0,
        }
        frame_persons.append(person)

    # 오래 끊긴 track 정리
    dead = [tid for tid, tr in tracks.items() if t - tr.get("last_t", t) > TRACK_MAX_GAP_SEC * 4]
    for tid in dead:
        tracks.pop(tid, None)

    return frame_persons, tracks, next_track_id


def pose_limb_motion_score(p):
    # pose가 있으면 손목/팔꿈치/발목의 confidence와 box 대비 움직임 가능성을 간단 점수화.
    # 실제 velocity는 track 간 keypoint 연결이 필요하므로 여기서는 keypoint 신뢰도 보조 신호로만 사용.
    pose = p.get("pose")
    if not pose or pose.get("xy") is None:
        return 0.0
    conf = pose.get("conf")
    if conf is None:
        return 0.15
    try:
        conf = np.array(conf, dtype=np.float32)
        # COCO pose: elbows 7,8 / wrists 9,10 / ankles 15,16
        idx = [7, 8, 9, 10, 15, 16]
        vals = [conf[i] for i in idx if i < len(conf)]
        return clamp(float(np.mean(vals)) if vals else 0.0, 0.0, 1.0)
    except Exception:
        return 0.0


## 7. fight / stalking 강화 분석

두 label 모두 같은 track-pair 구조를 사용합니다.

- fight: 가까움 + 접촉/겹침 + 급격한 움직임 + 상대 움직임 + pose 보조
- stalking: 접촉 없이 일정 거리 유지 + 같은 방향 이동 + 거리 안정성 + 지속성

점수는 전체 평균이 아니라 sliding window의 최대 점수를 사용합니다.


In [9]:
def pair_record(fr, p1, p2):
    t = fr["t"]
    c1 = np.array(p1["center"], dtype=np.float32)
    c2 = np.array(p2["center"], dtype=np.float32)
    v1 = np.array(p1.get("velocity", [0, 0]), dtype=np.float32)
    v2 = np.array(p2.get("velocity", [0, 0]), dtype=np.float32)
    b1, b2 = p1["box"], p2["box"]
    _, h1 = box_wh(b1)
    _, h2 = box_wh(b2)
    scale = max(20.0, (h1 + h2) / 2.0)
    dist = float(np.linalg.norm(c1 - c2))
    dist_norm = dist / scale
    iou = box_iou(b1, b2)
    speed1 = float(np.linalg.norm(v1) / scale)
    speed2 = float(np.linalg.norm(v2) / scale)
    rel_speed = float(np.linalg.norm(v1 - v2) / scale)
    dot = float(np.dot(v1, v2))
    denom = float(np.linalg.norm(v1) * np.linalg.norm(v2))
    cos_sim = dot / denom if denom > 1e-6 else 0.0
    pair_box = union_box([b1, b2])
    limb = max(pose_limb_motion_score(p1), pose_limb_motion_score(p2))

    return {
        "t": float(t),
        "pair": tuple(sorted([int(p1["track_id"]), int(p2["track_id"])])),
        "pair_box": pair_box,
        "dist_norm": float(dist_norm),
        "iou": float(iou),
        "close": bool(dist_norm < 1.0),
        "very_close": bool(dist_norm < 0.60),
        "contact": bool(iou > 0.015 or dist_norm < 0.55),
        "medium_distance": bool(0.75 <= dist_norm <= 3.00),
        "speed1": float(speed1),
        "speed2": float(speed2),
        "max_speed": float(max(speed1, speed2)),
        "rel_speed": float(rel_speed),
        "same_dir": bool(cos_sim > 0.55 and max(speed1, speed2) > 0.08),
        "opposite_or_cross": bool(cos_sim < -0.15 and max(speed1, speed2) > 0.08),
        "cos_sim": float(cos_sim),
        "limb_motion": float(limb),
    }


def build_pair_records(frames):
    pair_records = defaultdict(list)
    for fr in frames:
        persons = fr.get("persons", [])
        if len(persons) < 2:
            continue
        # confidence 상위 6명까지만 조합 폭증 방지
        persons = sorted(persons, key=lambda p: p.get("conf", 0), reverse=True)[:6]
        for p1, p2 in combinations(persons, 2):
            rec = pair_record(fr, p1, p2)
            pair_records[rec["pair"]].append(rec)
    for k in list(pair_records.keys()):
        pair_records[k] = sorted(pair_records[k], key=lambda r: r["t"])
    return pair_records


def score_pair_window(recs, start, end):
    ws = [r for r in recs if start <= r["t"] < end]
    if len(ws) < MIN_WINDOW_OBS:
        return None
    n = len(ws)
    close_ratio = safe_div(sum(r["close"] for r in ws), n)
    very_close_ratio = safe_div(sum(r["very_close"] for r in ws), n)
    contact_ratio = safe_div(sum(r["contact"] for r in ws), n)
    medium_ratio = safe_div(sum(r["medium_distance"] for r in ws), n)
    same_dir_ratio = safe_div(sum(r["same_dir"] for r in ws), n)
    opposite_ratio = safe_div(sum(r["opposite_or_cross"] for r in ws), n)
    speeds = np.array([r["max_speed"] for r in ws], dtype=np.float32)
    rels = np.array([r["rel_speed"] for r in ws], dtype=np.float32)
    dists = np.array([r["dist_norm"] for r in ws], dtype=np.float32)
    limb = max([r["limb_motion"] for r in ws] + [0.0])
    speed75 = float(np.percentile(speeds, 75)) if len(speeds) else 0.0
    rel75 = float(np.percentile(rels, 75)) if len(rels) else 0.0
    sudden_motion = clamp((speed75 - 0.08) / 0.55)
    rel_motion = clamp((rel75 - 0.05) / 0.60)
    duration_ratio = clamp((ws[-1]["t"] - ws[0]["t"]) / max(1e-3, WINDOW_SEC))
    dist_stability = clamp(1.0 - safe_div(float(np.std(dists)), max(0.25, float(np.mean(dists))), 0.0))

    # fight: 단순 가까움만으로는 부족. 접촉/초근접 + 갑작스러운 움직임 조합을 요구.
    fight_raw = (
        0.24 * close_ratio +
        0.22 * contact_ratio +
        0.12 * very_close_ratio +
        0.20 * sudden_motion +
        0.12 * rel_motion +
        0.06 * opposite_ratio +
        0.04 * limb
    )
    if contact_ratio < 0.10 and very_close_ratio < 0.15:
        fight_raw *= 0.72
    if sudden_motion < 0.10 and rel_motion < 0.10:
        fight_raw *= 0.78

    # stalking: 접촉 없이 중거리 유지 + 같은 방향 + 거리 안정성이 중요.
    no_contact_ratio = 1.0 - contact_ratio
    stalking_raw = (
        0.28 * medium_ratio +
        0.22 * same_dir_ratio +
        0.20 * dist_stability +
        0.16 * duration_ratio +
        0.14 * no_contact_ratio
    )
    if contact_ratio > 0.18 or very_close_ratio > 0.25:
        stalking_raw *= 0.55
    if sudden_motion > 0.38 and contact_ratio > 0.10:
        stalking_raw *= 0.60
    if medium_ratio < 0.25:
        stalking_raw *= 0.75

    return {
        "pair": ws[0]["pair"],
        "window_start": float(start),
        "window_end": float(end),
        "obs": int(n),
        "close_ratio": float(close_ratio),
        "very_close_ratio": float(very_close_ratio),
        "contact_ratio": float(contact_ratio),
        "medium_distance_ratio": float(medium_ratio),
        "same_dir_ratio": float(same_dir_ratio),
        "opposite_ratio": float(opposite_ratio),
        "speed75": float(speed75),
        "rel_speed75": float(rel75),
        "sudden_motion": float(sudden_motion),
        "rel_motion": float(rel_motion),
        "dist_stability": float(dist_stability),
        "limb_motion": float(limb),
        "fight_score": float(clamp(fight_raw)),
        "stalking_score": float(clamp(stalking_raw)),
    }


def analyze_pair_windows(frames):
    pair_records = build_pair_records(frames)
    all_windows = []
    best_fight = None
    best_stalking = None

    for pair, recs in pair_records.items():
        if len(recs) < MIN_PAIR_OBS:
            continue
        t_min = recs[0]["t"]
        t_max = recs[-1]["t"]
        start = t_min
        while start <= t_max:
            end = start + WINDOW_SEC
            w = score_pair_window(recs, start, end)
            if w is not None:
                all_windows.append(w)
                if best_fight is None or w["fight_score"] > best_fight["fight_score"]:
                    best_fight = w
                if best_stalking is None or w["stalking_score"] > best_stalking["stalking_score"]:
                    best_stalking = w
            start += WINDOW_STEP_SEC

    fight_score = float(best_fight["fight_score"]) if best_fight else 0.0
    stalking_score = float(best_stalking["stalking_score"]) if best_stalking else 0.0

    # 둘의 충돌 보정: 싸움의 접촉/급격 움직임이 충분하면 stalking 감쇠.
    if best_fight:
        if best_fight["contact_ratio"] > 0.20 and best_fight["sudden_motion"] > 0.25:
            stalking_score *= 0.55
        elif best_fight["very_close_ratio"] > 0.35 and best_fight["rel_motion"] > 0.25:
            stalking_score *= 0.70

    # 반대로 접촉이 거의 없고 안정 추적이 강하면 fight 약간 감쇠.
    if best_stalking:
        if best_stalking["contact_ratio"] < 0.08 and best_stalking["same_dir_ratio"] > 0.45 and best_stalking["dist_stability"] > 0.55:
            fight_score *= 0.82

    return {
        "pair_count": int(len(pair_records)),
        "window_count": int(len(all_windows)),
        "fight_score": float(clamp(fight_score)),
        "stalking_score": float(clamp(stalking_score)),
        "best_fight": best_fight,
        "best_stalking": best_stalking,
        "top_windows": sorted(all_windows, key=lambda x: max(x["fight_score"], x["stalking_score"]), reverse=True)[:8],
    }


## 8. 이륜차 / 전동 킥보드 / 인도 주행 분석

COCO의 `bicycle`, `motorcycle`과 YOLO-World/전용 모델의 `scooter`, `kickboard`, `electric scooter`를 함께 사용합니다.

인도 주행은 `이륜차 또는 킥보드 + 탑승자 + 인도 영역`이 같이 있을 때만 강하게 올라갑니다.


In [10]:
def get_twowheelers(objects):
    out = []
    for o in objects:
        cname = o.get("class_name", "")
        src = o.get("source", "")
        is_coco_tw = src == "coco" and o.get("class_id") in COCO_TWOWHEELER_CLASSES
        is_name_tw = is_twowheeler_name(cname)
        if (is_coco_tw or is_name_tw) and o.get("conf", 0) >= TWOWHEELER_CONF:
            oo = dict(o)
            oo["is_kickboard"] = bool(is_kickboard_name(cname))
            out.append(oo)
    return out


def rider_association(persons, twowheelers):
    pairs = []
    for tw in twowheelers:
        tw_box = tw["box"]
        tw_exp = expand_box(tw_box, 0.45)
        best = None
        best_score = 0.0
        for p in persons:
            c = np.array(p["center"], dtype=np.float32)
            f = np.array(p["foot"], dtype=np.float32)
            iou = box_iou(p["box"], tw_exp)
            inside = point_in_box(c, tw_exp) or point_in_box(f, tw_exp)
            dist = float(np.linalg.norm(c - box_center(tw_box)))
            _, ph = box_wh(p["box"])
            dist_score = math.exp(-dist / max(30.0, ph * 2.0))
            score = max(iou, 0.30 if inside else 0.0, 0.45 * dist_score)
            if score > best_score:
                best_score = score
                best = p
        if best is not None and best_score >= 0.18:
            pairs.append({"tw": tw, "person": best, "score": float(best_score)})
    return pairs


def analyze_twowheeler_and_sidewalk(frames):
    total = len(frames)
    if total == 0:
        return {
            "twowheeler_score": 0.0,
            "sidewalk_riding_score": 0.0,
            "electric_kickboard_score": 0.0,
            "rider_frame_ratio": 0.0,
            "sidewalk_ratio": 0.0,
            "kickboard_ratio": 0.0,
            "sidewalk_status": "unknown",
            "evidence": "no_frames",
        }

    tw_frames = 0
    rider_frames = 0
    sidewalk_frames = 0
    surface_known_frames = 0
    kickboard_frames = 0
    kickboard_rider_frames = 0
    kickboard_sidewalk_frames = 0
    class_counter = Counter()

    for fr in frames:
        tws = get_twowheelers(fr.get("objects", []))
        persons = fr.get("persons", [])
        if tws:
            tw_frames += 1
            for tw in tws:
                class_counter[normalize_name(tw.get("class_name"))] += 1
        pairs = rider_association(persons, tws)
        if pairs:
            rider_frames += 1

        any_kick = any(tw.get("is_kickboard") for tw in tws)
        if any_kick:
            kickboard_frames += 1
        any_kick_rider = any(pair["tw"].get("is_kickboard") for pair in pairs)
        if any_kick_rider:
            kickboard_rider_frames += 1

        # sidewalk 판정: 탑승자 pair가 있을 때 vehicle bottom 또는 rider foot이 sidewalk인지 확인
        frame_surface_known = False
        frame_sidewalk = False
        frame_kick_sidewalk = False
        for pair in pairs:
            tw = pair["tw"]
            p = pair["person"]
            tw_foot = box_foot(tw["box"])
            rider_foot = np.array(p["foot"], dtype=np.float32)
            s1 = surface_at_point(fr.get("seg_mask"), tw_foot)
            s2 = surface_at_point(fr.get("seg_mask"), rider_foot)
            if s1 != "unknown" or s2 != "unknown":
                frame_surface_known = True
            if s1 == "sidewalk" or s2 == "sidewalk":
                frame_sidewalk = True
                if tw.get("is_kickboard"):
                    frame_kick_sidewalk = True
        if frame_surface_known:
            surface_known_frames += 1
        if frame_sidewalk:
            sidewalk_frames += 1
        if frame_kick_sidewalk:
            kickboard_sidewalk_frames += 1

    tw_presence_ratio = safe_div(tw_frames, total)
    rider_ratio = safe_div(rider_frames, total)
    sidewalk_ratio = safe_div(sidewalk_frames, max(1, surface_known_frames), 0.0)
    kick_ratio = safe_div(kickboard_frames, total)
    kick_rider_ratio = safe_div(kickboard_rider_frames, total)
    kick_sidewalk_ratio = safe_div(kickboard_sidewalk_frames, max(1, surface_known_frames), 0.0)

    # segmentation/polygon 근거가 없으면 sidewalk score를 보수적으로 0에 가깝게 둠.
    surface_support = clamp(surface_known_frames / max(1, min(total, 8)))
    sidewalk_score = clamp((0.35 * rider_ratio + 0.65 * sidewalk_ratio) * surface_support)

    # 전동 킥보드는 world/전용 모델이 있으면 class 기반으로 잡고, 탑승자 association이 있으면 강화.
    electric_kickboard_score = clamp(0.45 * kick_ratio + 0.40 * kick_rider_ratio + 0.15 * kick_sidewalk_ratio)
    twowheeler_score = clamp(0.50 * tw_presence_ratio + 0.50 * rider_ratio)

    if surface_known_frames == 0:
        sidewalk_status = "unknown"
    elif sidewalk_ratio >= 0.35 and rider_ratio >= 0.10:
        sidewalk_status = "sidewalk_riding_candidate"
    elif rider_ratio >= 0.10:
        sidewalk_status = "road_or_non_sidewalk"
    else:
        sidewalk_status = "twowheeler_no_rider_or_unclear"

    evidence = (
        f"tw_frames={tw_frames}, rider_frames={rider_frames}, surface_known={surface_known_frames}, "
        f"sidewalk_frames={sidewalk_frames}, kickboard_frames={kickboard_frames}, "
        f"kickboard_rider_frames={kickboard_rider_frames}, classes={dict(class_counter.most_common(5))}"
    )

    return {
        "twowheeler_score": float(twowheeler_score),
        "sidewalk_riding_score": float(sidewalk_score),
        "electric_kickboard_score": float(electric_kickboard_score),
        "rider_frame_ratio": float(rider_ratio),
        "sidewalk_ratio": float(sidewalk_ratio),
        "surface_known_frames": int(surface_known_frames),
        "kickboard_ratio": float(kick_ratio),
        "kickboard_rider_ratio": float(kick_rider_ratio),
        "kickboard_sidewalk_ratio": float(kick_sidewalk_ratio),
        "sidewalk_status": sidewalk_status,
        "twowheeler_classes": dict(class_counter.most_common(10)),
        "evidence": evidence,
    }


## 9. 헬멧 미착용 분석

전용 helmet/no_helmet 모델이 있으면 가장 좋습니다. 없을 때는 YOLO-World의 `helmet/no helmet` class를 보조로 사용하지만, 오탐을 줄이기 위해 **탑승자 주변에서 반복적으로 관측될 때만** 점수를 올립니다.


In [11]:
def crop_box(frame, box, pad=0.10):
    H, W = frame.shape[:2]
    b = expand_box(box, pad, frame.shape)
    x1, y1, x2, y2 = map(int, b)
    if x2 <= x1 or y2 <= y1:
        return None
    return frame[y1:y2, x1:x2].copy()


def head_box_from_person(person):
    x1, y1, x2, y2 = box_xyxy(person["box"])
    h = y2 - y1
    return [x1, y1, x2, y1 + h * 0.45]


def helmet_crop_predict(frame, person):
    if helmet_model is None:
        return None
    hb = head_box_from_person(person)
    crop = crop_box(frame, hb, pad=0.20)
    if crop is None or crop.size == 0:
        return None
    try:
        res = helmet_model.predict(crop, imgsz=320, conf=HELMET_CONF, verbose=False)[0]
        objs = yolo_result_to_objects(res, names=helmet_model.names, source="helmet_model_crop")
        best = None
        for o in objs:
            n = normalize_name(o.get("class_name"))
            if is_helmet_name(n) or is_no_helmet_name(n):
                if best is None or o["conf"] > best["conf"]:
                    best = o
        if best is None:
            return None
        return {"status": "no_helmet" if is_no_helmet_name(best["class_name"]) else "helmet", "conf": float(best["conf"]), "source": "helmet_model"}
    except Exception:
        return None


def world_helmet_near_head(objects, person):
    head = head_box_from_person(person)
    head_exp = expand_box(head, 0.45)
    best = None
    for o in objects:
        n = normalize_name(o.get("class_name"))
        if not (is_helmet_name(n) or is_no_helmet_name(n)):
            continue
        if box_iou(o["box"], head_exp) > 0.01 or point_in_box(box_center(o["box"]), head_exp):
            if best is None or o["conf"] > best["conf"]:
                best = o
    if best is None:
        return None
    return {"status": "no_helmet" if is_no_helmet_name(best["class_name"]) else "helmet", "conf": float(best["conf"]), "source": best.get("source", "world")}


def analyze_helmet(frames):
    rider_obs = 0
    helmet_obs = 0
    no_helmet_obs = 0
    unknown_obs = 0
    source_counter = Counter()

    for fr in frames:
        tws = get_twowheelers(fr.get("objects", []))
        persons = fr.get("persons", [])
        pairs = rider_association(persons, tws)
        if not pairs:
            continue
        # 프레임당 탑승자 최대 2명까지만 확인
        for pair in pairs[:2]:
            rider_obs += 1
            person = pair["person"]
            pred = helmet_crop_predict(fr.get("frame"), person) if fr.get("frame") is not None else None
            if pred is None:
                pred = world_helmet_near_head(fr.get("objects", []), person)
            if pred is None:
                unknown_obs += 1
                continue
            source_counter[pred["source"]] += 1
            if pred["status"] == "no_helmet":
                no_helmet_obs += 1
            elif pred["status"] == "helmet":
                helmet_obs += 1
            else:
                unknown_obs += 1

    valid_obs = helmet_obs + no_helmet_obs
    no_helmet_ratio = safe_div(no_helmet_obs, valid_obs, 0.0)
    valid_ratio = safe_div(valid_obs, rider_obs, 0.0)

    # 전용 모델이면 더 신뢰, world만 있으면 조금 보수적으로.
    model_bonus = 1.0 if helmet_model is not None else 0.78
    no_helmet_score = clamp(no_helmet_ratio * valid_ratio * model_bonus)

    if rider_obs == 0:
        status = "no_rider"
    elif valid_obs == 0:
        status = "unknown"
    elif no_helmet_ratio >= 0.60:
        status = "no_helmet_candidate"
    elif helmet_obs > no_helmet_obs:
        status = "helmet_candidate"
    else:
        status = "unknown"

    return {
        "no_helmet_score": float(no_helmet_score),
        "helmet_status": status,
        "rider_obs": int(rider_obs),
        "helmet_obs": int(helmet_obs),
        "no_helmet_obs": int(no_helmet_obs),
        "helmet_unknown_obs": int(unknown_obs),
        "helmet_valid_ratio": float(valid_ratio),
        "helmet_sources": dict(source_counter),
        "evidence": f"rider_obs={rider_obs}, helmet={helmet_obs}, no_helmet={no_helmet_obs}, unknown={unknown_obs}, sources={dict(source_counter)}",
    }


## 10. 침입 분석

침입은 오탐이 쉽게 늘어나므로, 기본적으로 다음 중 하나가 있어야 강하게 인정합니다.

1. `INTRUSION_LINES`에 사용자가 지정한 경계선을 사람이 foot point로 crossing
2. `RESTRICTED_POLYGONS`에 지정한 제한구역으로 foot point가 진입
3. YOLO-World가 gate/fence/railing/door를 잡고, 그 근처에서 foot point crossing/진입이 반복 관측

사용자 지정 선이나 polygon이 없으면 `intrusion`은 매우 보수적으로 처리됩니다.


In [12]:
def get_barriers(objects):
    barriers = []
    for o in objects:
        if is_barrier_name(o.get("class_name", "")) and o.get("conf", 0) >= WORLD_CONF:
            barriers.append(o)
    return barriers


def analyze_intrusion(frames):
    # track별 foot history 수집
    histories = defaultdict(list)
    barrier_frames = 0
    barrier_near_events = 0
    explicit_cross = 0
    restricted_enter = 0
    examples = []

    for fr in frames:
        barriers = get_barriers(fr.get("objects", []))
        if barriers:
            barrier_frames += 1
        barrier_boxes = [expand_box(b["box"], 0.35) for b in barriers]
        for p in fr.get("persons", []):
            tid = int(p["track_id"])
            foot = np.array(p["foot"], dtype=np.float32)
            histories[tid].append((float(fr["t"]), foot, p))
            if any(point_in_box(foot, bb) for bb in barrier_boxes):
                barrier_near_events += 1

    for tid, hist in histories.items():
        hist = sorted(hist, key=lambda x: x[0])
        for (t0, f0, _), (t1, f1, p1) in zip(hist[:-1], hist[1:]):
            for line in INTRUSION_LINES:
                if segment_crossed(f0, f1, line["p1"], line["p2"]):
                    explicit_cross += 1
                    if len(examples) < 5:
                        examples.append({"track_id": tid, "t": t1, "type": "line_cross", "name": line.get("name", "line")})
            for area in RESTRICTED_POLYGONS:
                prev_in = point_in_poly(f0, area.get("polygon", []))
                cur_in = point_in_poly(f1, area.get("polygon", []))
                if (not prev_in) and cur_in:
                    restricted_enter += 1
                    if len(examples) < 5:
                        examples.append({"track_id": tid, "t": t1, "type": "restricted_enter", "name": area.get("name", "area")})

    track_count = len(histories)
    explicit_support = explicit_cross + restricted_enter
    barrier_support = safe_div(barrier_near_events, max(1, sum(len(h) for h in histories)), 0.0)
    barrier_presence = safe_div(barrier_frames, max(1, len(frames)), 0.0)

    # 명시 경계/제한구역이 있으면 강하게, barrier만 있으면 약하게.
    score = 0.0
    if explicit_support > 0:
        score = clamp(0.55 + 0.15 * explicit_support + 0.15 * barrier_presence)
    elif barrier_presence > 0.10 and barrier_support > 0.08:
        score = clamp(0.25 * barrier_presence + 0.35 * barrier_support)

    status = "unknown"
    if explicit_support > 0:
        status = "intrusion_candidate"
    elif barrier_presence > 0.10:
        status = "barrier_near_motion_only"

    return {
        "intrusion_score": float(score),
        "intrusion_status": status,
        "track_count": int(track_count),
        "barrier_frames": int(barrier_frames),
        "barrier_near_events": int(barrier_near_events),
        "explicit_cross": int(explicit_cross),
        "restricted_enter": int(restricted_enter),
        "examples": examples,
        "evidence": f"explicit_cross={explicit_cross}, restricted_enter={restricted_enter}, barrier_frames={barrier_frames}, barrier_near_events={barrier_near_events}",
    }


## 11. final label 1개 선택

최종 표에는 여러 label을 동시에 찍지 않습니다.

모든 label 후보를 점수화한 뒤,

1. gate 조건을 통과한 후보만 남기고
2. label별 가중치를 곱하고
3. 충돌 보정을 적용하고
4. 가장 높은 후보 하나만 `final_label`로 선택합니다.


In [13]:
def window_text(w):
    if not w:
        return "-"
    return f"{w['window_start']:.1f}-{w['window_end']:.1f}s pair={w['pair']}"


def build_evidence_text(final_label, pair_res, tw_res, helmet_res, intrusion_res):
    if final_label == "fight":
        b = pair_res.get("best_fight")
        if not b:
            return "fight evidence unavailable"
        return (
            f"fight_window={window_text(b)}, close={b['close_ratio']:.2f}, "
            f"contact={b['contact_ratio']:.2f}, sudden={b['sudden_motion']:.2f}, rel={b['rel_motion']:.2f}"
        )
    if final_label == "stalking":
        b = pair_res.get("best_stalking")
        if not b:
            return "stalking evidence unavailable"
        return (
            f"stalking_window={window_text(b)}, medium={b['medium_distance_ratio']:.2f}, "
            f"same_dir={b['same_dir_ratio']:.2f}, stable={b['dist_stability']:.2f}, contact={b['contact_ratio']:.2f}"
        )
    if final_label in ["sidewalk_riding", "electric_kickboard"]:
        return tw_res.get("evidence", "twowheeler evidence unavailable")
    if final_label == "no_helmet":
        return helmet_res.get("evidence", "helmet evidence unavailable")
    if final_label == "intrusion":
        return intrusion_res.get("evidence", "intrusion evidence unavailable")
    return "no abnormal label passed gates"


def choose_final_label(pair_res, tw_res, helmet_res, intrusion_res):
    raw = {
        "fight": float(pair_res.get("fight_score", 0.0)),
        "stalking": float(pair_res.get("stalking_score", 0.0)),
        "sidewalk_riding": float(tw_res.get("sidewalk_riding_score", 0.0)),
        "no_helmet": float(helmet_res.get("no_helmet_score", 0.0)),
        "electric_kickboard": float(tw_res.get("electric_kickboard_score", 0.0)),
        "intrusion": float(intrusion_res.get("intrusion_score", 0.0)),
    }

    best_fight = pair_res.get("best_fight") or {}
    best_stalking = pair_res.get("best_stalking") or {}

    # 추가 충돌 보정
    adjusted = dict(raw)
    if best_fight.get("contact_ratio", 0) > 0.20 and best_fight.get("sudden_motion", 0) > 0.25:
        adjusted["stalking"] *= 0.60
    if best_stalking.get("contact_ratio", 0) < 0.06 and best_stalking.get("same_dir_ratio", 0) > 0.50:
        adjusted["fight"] *= 0.88

    # 전동킥보드와 인도주행이 동시에 뜨면 더 구체적인 전동킥보드를 약간 우대하되,
    # sidewalk 근거가 명확하면 sidewalk_riding도 경쟁 가능하게 둠.
    if adjusted["electric_kickboard"] > 0.35 and tw_res.get("sidewalk_riding_score", 0) > 0.40:
        adjusted["electric_kickboard"] *= 1.06

    candidates = []

    # fight gate
    if adjusted["fight"] >= LABEL_THRESHOLDS["fight"]:
        if best_fight and (best_fight.get("contact_ratio", 0) >= 0.10 or best_fight.get("very_close_ratio", 0) >= 0.18):
            candidates.append({"label": "fight", "raw_score": raw["fight"], "adjusted_score": adjusted["fight"]})

    # stalking gate
    if adjusted["stalking"] >= LABEL_THRESHOLDS["stalking"]:
        if best_stalking and best_stalking.get("contact_ratio", 1) <= 0.18 and best_stalking.get("medium_distance_ratio", 0) >= 0.25:
            candidates.append({"label": "stalking", "raw_score": raw["stalking"], "adjusted_score": adjusted["stalking"]})

    # sidewalk riding gate
    if adjusted["sidewalk_riding"] >= LABEL_THRESHOLDS["sidewalk_riding"]:
        if tw_res.get("rider_frame_ratio", 0) >= 0.10 and tw_res.get("surface_known_frames", 0) >= 3 and tw_res.get("sidewalk_ratio", 0) >= 0.30:
            candidates.append({"label": "sidewalk_riding", "raw_score": raw["sidewalk_riding"], "adjusted_score": adjusted["sidewalk_riding"]})

    # no helmet gate
    if adjusted["no_helmet"] >= LABEL_THRESHOLDS["no_helmet"]:
        if helmet_res.get("rider_obs", 0) >= 2 and helmet_res.get("no_helmet_obs", 0) >= 1 and helmet_res.get("helmet_valid_ratio", 0) >= 0.20:
            candidates.append({"label": "no_helmet", "raw_score": raw["no_helmet"], "adjusted_score": adjusted["no_helmet"]})

    # electric kickboard gate
    if adjusted["electric_kickboard"] >= LABEL_THRESHOLDS["electric_kickboard"]:
        if tw_res.get("kickboard_ratio", 0) >= 0.08 or tw_res.get("kickboard_rider_ratio", 0) >= 0.05:
            candidates.append({"label": "electric_kickboard", "raw_score": raw["electric_kickboard"], "adjusted_score": adjusted["electric_kickboard"]})

    # intrusion gate
    if adjusted["intrusion"] >= LABEL_THRESHOLDS["intrusion"]:
        if intrusion_res.get("explicit_cross", 0) > 0 or intrusion_res.get("restricted_enter", 0) > 0:
            candidates.append({"label": "intrusion", "raw_score": raw["intrusion"], "adjusted_score": adjusted["intrusion"]})

    for c in candidates:
        c["weighted_score"] = float(c["adjusted_score"] * LABEL_WEIGHTS.get(c["label"], 1.0))
        c["label_ko"] = LABEL_KO.get(c["label"], c["label"])

    candidates = sorted(candidates, key=lambda x: x["weighted_score"], reverse=True)

    if not candidates:
        final = {
            "final_label": "other",
            "final_label_ko": LABEL_KO["other"],
            "final_raw_score": 0.0,
            "final_adjusted_score": 0.0,
            "final_weighted_score": 0.0,
            "candidates": [],
            "raw_scores": raw,
            "adjusted_scores": adjusted,
        }
    else:
        top = candidates[0]
        final = {
            "final_label": top["label"],
            "final_label_ko": top["label_ko"],
            "final_raw_score": float(top["raw_score"]),
            "final_adjusted_score": float(top["adjusted_score"]),
            "final_weighted_score": float(top["weighted_score"]),
            "candidates": candidates,
            "raw_scores": raw,
            "adjusted_scores": adjusted,
        }
    return final


## 12. 영상 1개 처리 함수


In [14]:
def process_video(video_path):
    video_path = Path(video_path)
    info = get_video_info(video_path)
    if not info.get("ok"):
        raise RuntimeError(f"영상 정보를 읽을 수 없습니다: {video_path}")

    frames = []
    tracks = {}
    next_track_id = 1

    for item in iter_sampled_frames(video_path, ANALYZE_FPS, MAX_SECONDS_PER_VIDEO):
        frame = item["frame"]
        sampled_idx = item["sampled_idx"]
        t = item["t"]

        objects = detect_all_objects(frame)
        person_dets = detect_persons_from_objects(objects)

        if USE_POSE and pose_model is not None and sampled_idx % POSE_EVERY_N_SAMPLED_FRAMES == 0:
            pose_dets = yolo_pose_persons(frame)
            person_dets = attach_pose_to_persons(person_dets, pose_dets)
        else:
            for p in person_dets:
                p["pose"] = None

        persons, tracks, next_track_id = update_person_tracks(person_dets, tracks, next_track_id, t)

        seg_mask = None
        if (seg_model is not None) and (sampled_idx % SEG_EVERY_N_SAMPLED_FRAMES == 0):
            seg_mask = segment_frame_surface(frame)

        frames.append({
            "video_name": video_path.name,
            "frame_idx": int(item["frame_idx"]),
            "sampled_idx": int(sampled_idx),
            "t": float(t),
            "persons": persons,
            "objects": objects,
            "object_count": int(len(objects)),
            "seg_mask": seg_mask,
            # helmet crop용. JSON 저장 전에 제거함.
            "frame": frame,
        })

    pair_res = analyze_pair_windows(frames)
    tw_res = analyze_twowheeler_and_sidewalk(frames)
    helmet_res = analyze_helmet(frames)
    intrusion_res = analyze_intrusion(frames)
    final_res = choose_final_label(pair_res, tw_res, helmet_res, intrusion_res)
    evidence = build_evidence_text(final_res["final_label"], pair_res, tw_res, helmet_res, intrusion_res)

    # JSON 저장용으로 frame/seg_mask 제거하고 가벼운 frame summary만 저장
    frame_summaries = []
    for fr in frames:
        obj_classes = Counter(normalize_name(o.get("class_name")) for o in fr.get("objects", []))
        frame_summaries.append({
            "frame_idx": fr["frame_idx"],
            "sampled_idx": fr["sampled_idx"],
            "t": fr["t"],
            "person_count": len(fr.get("persons", [])),
            "object_count": fr.get("object_count", 0),
            "object_classes": dict(obj_classes.most_common(10)),
            "persons": [
                {
                    "track_id": p["track_id"],
                    "box": p["box"],
                    "conf": p.get("conf", 0),
                    "center": p.get("center"),
                    "foot": p.get("foot"),
                    "speed_px_s": p.get("speed_px_s", 0),
                }
                for p in fr.get("persons", [])
            ],
            "twowheelers": [
                {"box": o["box"], "conf": o["conf"], "class_name": o["class_name"], "source": o.get("source")}
                for o in get_twowheelers(fr.get("objects", []))
            ],
        })

    person_track_ids = sorted({p["track_id"] for fr in frames for p in fr.get("persons", [])})
    max_persons = max([len(fr.get("persons", [])) for fr in frames] + [0])

    detail = {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "video_info": info,
        "final": final_res,
        "evidence": evidence,
        "pair_result": pair_res,
        "twowheeler_sidewalk_result": tw_res,
        "helmet_result": helmet_res,
        "intrusion_result": intrusion_res,
        "person_track_count": len(person_track_ids),
        "max_persons_in_frame": int(max_persons),
        "frames": frame_summaries,
    }

    out_json = DETAIL_DIR / f"{video_path.stem}_ee_detail.json"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(detail, f, ensure_ascii=False, indent=2)

    row = {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "final_label": final_res["final_label"],
        "final_label_ko": final_res["final_label_ko"],
        "final_weighted_score": final_res["final_weighted_score"],
        "final_raw_score": final_res["final_raw_score"],
        "fight_score": pair_res.get("fight_score", 0.0),
        "stalking_score": pair_res.get("stalking_score", 0.0),
        "sidewalk_riding_score": tw_res.get("sidewalk_riding_score", 0.0),
        "no_helmet_score": helmet_res.get("no_helmet_score", 0.0),
        "electric_kickboard_score": tw_res.get("electric_kickboard_score", 0.0),
        "intrusion_score": intrusion_res.get("intrusion_score", 0.0),
        "rider_frame_ratio": tw_res.get("rider_frame_ratio", 0.0),
        "sidewalk_ratio": tw_res.get("sidewalk_ratio", 0.0),
        "helmet_status": helmet_res.get("helmet_status", "unknown"),
        "intrusion_status": intrusion_res.get("intrusion_status", "unknown"),
        "person_track_count": len(person_track_ids),
        "max_persons_in_frame": int(max_persons),
        "pair_count": pair_res.get("pair_count", 0),
        "best_fight_window": window_text(pair_res.get("best_fight")),
        "best_stalking_window": window_text(pair_res.get("best_stalking")),
        "evidence": evidence,
        "json_path": str(out_json),
    }
    return row, detail


## 13. 배치 처리

`VIDEO_ROOT` 아래의 모든 영상 파일을 재귀적으로 찾습니다.


In [15]:
def collect_video_files(root):
    root = Path(root)
    if root.is_file() and root.suffix.lower() in VIDEO_EXTS:
        return [root]
    if not root.exists():
        return []
    files = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
            # 출력 폴더 내부 영상은 제외
            if str(OUTPUT_ROOT) in str(p):
                continue
            files.append(p)
    return sorted(files)

video_files = collect_video_files(VIDEO_ROOT)
print("video count:", len(video_files))
for p in video_files[:10]:
    print("-", p)
if len(video_files) > 10:
    print("...")


video count: 22
- C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c1.mp4
- C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c2.mp4
- C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c1.mp4
- C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c2.mp4
- C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c1.mp4
- C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c2.mp4
- C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c1.mp4
- C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c2.mp4
- C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c1.mp4
- C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c2.mp4
...


In [16]:
results = []
failed = []

for vp in tqdm(video_files, desc="ee integrated batch"):
    try:
        row, detail = process_video(vp)
        results.append(row)
        print(f"OK {vp.name} -> {row['final_label']} ({row['final_label_ko']}) score={row['final_weighted_score']:.3f}")
    except Exception as e:
        failed.append({
            "video_name": Path(vp).name,
            "video_path": str(vp),
            "error": repr(e),
            "traceback": traceback.format_exc(),
        })
        print(f"FAIL {vp.name}: {e}")

print("success:", len(results), "/", len(video_files))
print("failed :", len(failed))


ee integrated batch:   0%|          | 0/22 [00:00<?, ?it/s]

FAIL lo_e1281_c1.mp4: object of type 'int' has no len()
FAIL lo_e1281_c2.mp4: object of type 'int' has no len()
FAIL fp_e1520_c1.mp4: object of type 'int' has no len()
FAIL fp_e1520_c2.mp4: object of type 'int' has no len()
FAIL ph_e2063_c1.mp4: object of type 'int' has no len()
FAIL ph_e2063_c2.mp4: object of type 'int' has no len()
FAIL ms_e3500_c1.mp4: object of type 'int' has no len()
OK ms_e3500_c2.mp4 -> other (기타) score=0.000
FAIL bs_e4387_c1.mp4: object of type 'int' has no len()
FAIL bs_e4387_c2.mp4: object of type 'int' has no len()
FAIL es_e3768_c1.mp4: object of type 'int' has no len()
FAIL es_e3768_c2.mp4: object of type 'int' has no len()
FAIL md_e2122_c1.mp4: object of type 'int' has no len()
FAIL md_e2122_c2.mp4: object of type 'int' has no len()
FAIL mp_e2730_c1.mp4: object of type 'int' has no len()
FAIL mp_e2730_c2.mp4: object of type 'int' has no len()
FAIL st_e4803_c1.mp4: object of type 'int' has no len()
FAIL st_e4803_c2.mp4: object of type 'int' has no len()
FAI

## 14. 최종 결과 표 출력 및 저장

표에는 `final_label`이 영상당 하나만 표시됩니다. 내부 후보 점수는 열로 같이 보여주되, 최종 판정은 한 개입니다.


In [17]:
summary_json = OUTPUT_ROOT / "ee_batch_summary.json"
summary_csv = OUTPUT_ROOT / "ee_batch_summary.csv"

batch = {
    "task": "ee_full_integrated_fight_stalking_enhanced",
    "description": "fight/stalking 강화 + sidewalk riding/no helmet/electric kickboard/intrusion 전체 label 유지",
    "video_root": str(VIDEO_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "total_video_count": len(video_files),
    "success_count": len(results),
    "failed_count": len(failed),
    "settings": {
        "ANALYZE_FPS": ANALYZE_FPS,
        "MAX_SECONDS_PER_VIDEO": MAX_SECONDS_PER_VIDEO,
        "WINDOW_SEC": WINDOW_SEC,
        "WINDOW_STEP_SEC": WINDOW_STEP_SEC,
        "USE_POSE": USE_POSE,
        "USE_YOLO_WORLD": USE_YOLO_WORLD,
        "USE_SEGFORMER": USE_SEGFORMER,
        "DETECT_MODEL_NAME": DETECT_MODEL_NAME,
        "POSE_MODEL_NAME": POSE_MODEL_NAME,
        "WORLD_MODEL_NAME": WORLD_MODEL_NAME,
        "HELMET_MODEL_PATH": str(HELMET_MODEL_PATH) if HELMET_MODEL_PATH else None,
        "KICKBOARD_MODEL_PATH": str(KICKBOARD_MODEL_PATH) if KICKBOARD_MODEL_PATH else None,
        "LABEL_WEIGHTS": LABEL_WEIGHTS,
        "LABEL_THRESHOLDS": LABEL_THRESHOLDS,
    },
    "label_ko": LABEL_KO,
    "results": results,
    "failed": failed,
}

with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(batch, f, ensure_ascii=False, indent=2)

if pd is None:
    print("pandas가 없어 표 출력이 제한됩니다. CMD에서 `pip install pandas` 후 다시 실행하세요.")
    print("summary json:", summary_json)
else:
    df = pd.DataFrame(results)
    if len(df) == 0:
        print("처리된 결과가 없습니다.")
    else:
        cols = [
            "video_name",
            "final_label",
            "final_label_ko",
            "final_weighted_score",
            "fight_score",
            "stalking_score",
            "sidewalk_riding_score",
            "no_helmet_score",
            "electric_kickboard_score",
            "intrusion_score",
            "rider_frame_ratio",
            "sidewalk_ratio",
            "helmet_status",
            "intrusion_status",
            "person_track_count",
            "max_persons_in_frame",
            "best_fight_window",
            "best_stalking_window",
            "evidence",
            "json_path",
        ]
        cols = [c for c in cols if c in df.columns]
        df_view = df[cols].copy()
        df_view = df_view.sort_values(
            by=["final_label", "final_weighted_score", "fight_score", "stalking_score"],
            ascending=[True, False, False, False],
        )
        df_view.to_csv(summary_csv, index=False, encoding="utf-8-sig")
        print("summary json:", summary_json)
        print("summary csv :", summary_csv)
        print("\nfinal_label 분포:")
        print(df_view["final_label"].value_counts())
        if display is not None:
            display(df_view)
        else:
            print(df_view.to_string(index=False))


summary json: C:\Users\우성한\outputs\prediction\ee\ee_batch_summary.json
summary csv : C:\Users\우성한\outputs\prediction\ee\ee_batch_summary.csv

final_label 분포:
final_label
other    1
Name: count, dtype: int64


,video_name,final_label,final_label_ko,final_weighted_score,fight_score,stalking_score,sidewalk_riding_score,no_helmet_score,electric_kickboard_score,intrusion_score,rider_frame_ratio,sidewalk_ratio,helmet_status,intrusion_status,person_track_count,max_persons_in_frame,best_fight_window,best_stalking_window,evidence,json_path
0,ms_e3500_c2.mp4,other,기타,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,no_rider,unknown,0,0,-,-,no abnormal label passed gates,C:\Users\우성한\outputs\prediction\ee\details\ms_...


## 15. label별 결과 빠르게 확인

아래 셀의 `TARGET_LABEL`만 바꾸면 특정 최종 label 영상만 볼 수 있습니다.


In [18]:
if pd is not None and "df" in globals() and len(df) > 0:
    # 예: "fight", "stalking", "sidewalk_riding", "no_helmet", "electric_kickboard", "intrusion", "other"
    TARGET_LABEL = "fight"
    sub = df[df["final_label"] == TARGET_LABEL].copy()
    if len(sub) == 0:
        print(f"final_label == {TARGET_LABEL!r} 인 영상이 없습니다.")
    else:
        view_cols = [
            "video_name",
            "final_label",
            "final_label_ko",
            "final_weighted_score",
            "fight_score",
            "stalking_score",
            "sidewalk_riding_score",
            "no_helmet_score",
            "electric_kickboard_score",
            "intrusion_score",
            "best_fight_window",
            "best_stalking_window",
            "evidence",
        ]
        view_cols = [c for c in view_cols if c in sub.columns]
        sub = sub[view_cols].sort_values("final_weighted_score", ascending=False)
        display(sub) if display is not None else print(sub.to_string(index=False))


final_label == 'fight' 인 영상이 없습니다.


## 16. 조정 포인트

이번 `ee`는 전체 이상행동 범위를 유지하면서, fight/stalking 쪽을 더 강하게 바꾼 버전입니다.

싸움이 너무 약하면:

```python
LABEL_THRESHOLDS["fight"] = 0.38
LABEL_WEIGHTS["fight"] = 1.45
```

스토킹이 싸움으로 너무 많이 밀리면:

```python
LABEL_THRESHOLDS["stalking"] = 0.42
LABEL_WEIGHTS["stalking"] = 1.25
```

인도 주행이 계속 `unknown`이면 SegFormer 설치 여부를 확인하거나, 직접 `SIDEWALK_POLYGONS`를 지정하는 방식이 가장 안정적입니다.

헬멧 미착용은 전용 helmet/no_helmet 모델이 없으면 구조적으로 한계가 있습니다. 그래도 이 노트북은 YOLO-World의 보조 탐지를 제한적으로 사용하지만, 확정 판정은 전용 모델이 있을 때 가장 안정적입니다.
